# 0. Initialiseer - refresh external data en import libraries

### 0.1 De df_dim_sensor tabel bevat de volgende informatie
- **locatie**            = de naam van de locatie waar de sensor is geplaats
- **device_name**        = de naam van de sensor die op de label van de fysieke sensor staat
- **datum_geplaatst**      = de datum waarop de sensor in de grond is geplaatst
- **datum_weggehaald**  = de datum waarop de sensor uit de grond is gehaald. Als deze datum na vandaag ligt, dan zit de sensor daar nog in de grond.
- **diepte_plaatsing**     = de diepte waarop de pinnen van de sensor in de grond zijn geplaats (15cm of 30cm diep)
- **leeftijd_plant**     = de leeftijd van de beplanting waar de de sensor is geplaats (0-1, 1-2 of 2-3 jaar)
- **soort_plant**        = de omschrijving van de beplanting waar de sensor is geplaatst
- **locatie_regio**      = de regio van de stad waar de sensor is geplaatst
- **extra_omschrijving** = een extra omschrijving van de locatie waar de sensor is geplaatst (bijv. in de berm, vlakbij bestrating, ..)
- **old**                = voor start van nieuwe project zijn er al sensoren geplaatst, deze hebben de indicatie 'old'. Van deze data weten we niet hoe betrouwbaar deze is.
- **device_name_org**    = voor het koppelen van de device_id's met de device_name's is de originele (= org) benaming aangepast om de koppeling te vereenvoudigen, daarom voor de zekerheid deze kolom wel behouden.
- **device_id**          = de id van de device zoals deze geregistreerd staat in de quantified portal. Middels de device id's kan de data via de quantified API opgehaald worden.

### 0.2 De df_fact_sensor tabel bevat de volgende informatie
- **gateway_receive_time**  = tijdstip waarop sensor permittivity waarde is ontvangen
- **device**                = device_id waar de permittivity waarde vandaan komt
- **value**                 = de gemeten permittivity waarde

### 0.3 De df_KNMI bevat de volgende informatie
- **YYYYMMDD**  = Datum (YYYY=jaar MM=maand DD=dag)
- **FG**        = Etmaalgemiddelde windsnelheid (in 0.1 m/s)
- (**TG**        = Etmaalgemiddelde temperatuur (in 0.1 graden Celsius)) --> <u>vervangen door Zusterhof</u>
- **SQ**        = Zonneschijnduur (in 0.1 uur) berekend uit de globale straling (-1 voor <0.05 uur)
- **DR**        = Duur van de neerslag (in 0.1 uur)
- (**RH**        = Etmaalsom van de neerslag (in 0.1 mm) (-1 voor <0.05 mm)) --> <u>vervangen door Zusterhof</u>
- **PG**        = Etmaalgemiddelde luchtdruk herleid tot zeeniveau (in 0.1 hPa) berekend uit 24 uurwaarden
- **NG**        = Etmaalgemiddelde bewolking (bedekkingsgraad van de bovenlucht in achtsten, 9=bovenlucht onzichtbaar)
- **UG**        = Etmaalgemiddelde relatieve vochtigheid (in procenten)
- **EV24**      = Referentiegewasverdamping (Makkink) (in 0.1 mm)

### 0.4 De df_weerstation_leiden bevat de volgende informatie
- **datum**         = Datum meetwaarde
- **temperatuur**   = Etmaalgemiddelde temperatuur (in 0.1 graden Celsius)
- **neerslag**      = Etmaalsom van de neerslag (in mm) (-1 voor <0.05 mm)

### Notes
- De temperatuur en neerslag wordt van weerstation **Zusterhof** in Leiden gebruikt. De overige weerdata vanuit het KNMI van weerstation **Schiphol**.
- De KNMI data is ook op uurniveau beschikbaar, maar deze is niet geschikt voor de analyse vanwege datakwaliteit, daarom gebruiken we KNMI data op dagniveau. 


In [21]:
# TODO:
# Stappenplan van acties voordat dit notebook gedraaid kan worden

In [22]:
# Import neccesary libraries
from custom_functions.data_transformations import anonymize_location, resample
import numpy as np
import plotly.express as px
from pandas.tseries.offsets import DateOffset

### 0. Refresh & Laad data:
- 1.1 Sensorinformatie vanuit google spreadsheet
- 1.2 Quantified data (= df_fact_sensor)
- 1.3 KNMI data (= df_KNMI)
- 1.4 Weerstation Leiden data (= df_weerstation_leiden)

In [23]:
# 0. Refresh & Laad data

refresh_quantify = False

if refresh_quantify:
    from custom_functions import refresh_quantified

from custom_functions import refresh_external_data
from custom_functions.data_loading import get_all_data

df_dim_sensor, df_fact_sensor, df_KNMI, df_weerstation_leiden = get_all_data()

### 1. Inspecteer data
- 1.1 Check data info, head and tail
- 1.2 Check data visuals

In [24]:
# 1.1 Check data info, head and tail

# df_dim_sensor
display(df_dim_sensor.info(5))
display(df_dim_sensor.head(5))
display(df_dim_sensor.tail(5))

# df_fact_sensor
display(df_fact_sensor.info(5))
display(df_fact_sensor.head(5))
display(df_fact_sensor.tail(5))

# df_KNMI
display(df_KNMI.info(5))
display(df_KNMI.head(5))
display(df_KNMI.tail(5))

# df_weerstation_leiden
display(df_weerstation_leiden.info(5))
display(df_weerstation_leiden.head(5))
display(df_weerstation_leiden.tail(5))

# 1.2 Check data visuals
df_visual = df_fact_sensor.copy()
df_visual.sort_values(by=['device', 'gateway_receive_time'], inplace=True)

fig = px.line(df_visual, x='gateway_receive_time', y='value', color='device', title='Time Series of all devices',
              labels={'value': 'Permittivity', 'gateway_receive_time': 'Time'},
              line_group='device', hover_name='device')

fig.show()

cutt_off = df_visual['gateway_receive_time'].max() - DateOffset(months=2)
fig = px.line(df_KNMI[df_KNMI['datum'] > cutt_off], x='datum', y='neerslag', title='Time Series of neerslag KNMI')

fig.show()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   locatie            42 non-null     object        
 1   device_name        42 non-null     object        
 2   datum_geplaatst    42 non-null     datetime64[ns]
 3   datum_weggehaald   42 non-null     datetime64[ns]
 4   soort_plant        42 non-null     object        
 5   leeftijd_plant     42 non-null     object        
 6   diepte_plaatsing   42 non-null     object        
 7   omgevingsfactoren  42 non-null     object        
 8   locatie_regio      42 non-null     object        
 9   old                42 non-null     object        
 10  current_location   42 non-null     object        
 11  device_name_org    42 non-null     object        
 12  device_id          42 non-null     int64         
dtypes: datetime64[ns](2), int64(1), object(10)
memory usage: 4.4+ KB


None

,locatie,device_name,datum_geplaatst,datum_weggehaald,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old,current_location,device_name_org,device_id
0,Arendshorst_30,FF2 0-0193,2023-08-02,2262-04-11,Boom,> 1 jaar,30 cm,groen,Noord,No,Yes,FF2 0-0193,418
1,Arendshorst (old)_30,FF2 0-0193,2023-07-04,2023-08-02,Boom,> 1 jaar,30 cm,groen,Noord,Yes,No,FF2 0-0193,418
2,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372
3,Bachstraat (2)_15,FF2 0-0147,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0147,395
4,Beethovenpark_30,FF1 0-0027,2023-07-04,2023-08-07,Plantenvak,0-1 jaar,30 cm,groen,Zuid,No,No,FF1 0-0027,54


,locatie,device_name,datum_geplaatst,datum_weggehaald,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old,current_location,device_name_org,device_id
37,Van Swietenstraat_15,FF2 0-0185,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Stevenshof en de mors,No,Yes,FF2 0-0185,410
38,Veluwemeerlaan (old)_30,FF2 0-0137,2023-07-04,2023-07-04,Heesters,> 1 jaar,30 cm,groen,Noord,Yes,No,FF2 0-0137,362
39,Vijf meilaan (1)_15,FF2 0-0102,2023-07-13,2262-04-11,Plantenvak,0-1 jaar,15 cm,bestrating,Zuid,No,Yes,FF2 0-0102,208
40,Vijf meilaan (2)_30,FF1 0-0004,2023-07-04,2262-04-11,Plantenvak,0-1 jaar,30 cm,bestrating,Zuid,No,Yes,FF1 0-0004,93
41,Zeemanlaan_15,FF2 0-0176,2023-07-04,2023-08-07,Plant en boom,> 1 jaar,15 cm,bestrating,Zuid,Yes,No,FF2 0-0176,399


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61296 entries, 0 to 61295
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   gateway_receive_time  61296 non-null  datetime64[ns]
 1   device                61296 non-null  int64         
 2   value                 61296 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.4 MB


...

None

,datum,temperatuur,neerslag
0,2022-02-19,6.3,9.3
1,2022-02-20,8.1,22.2
2,2022-02-21,7.0,1.8
3,2022-02-22,7.8,1.5
4,2022-02-23,8.3,0.0


,datum,temperatuur,neerslag
971,2024-11-08,5.3,0.0
972,2024-11-09,6.8,1.2
973,2024-11-10,9.3,0.0
974,2024-11-11,11.2,3.0
975,2024-11-12,9.2,0.0


c:\Users\JSpan\AppData\Local\anaconda3\envs\waterleiden\Lib\site-packages\_plotly_utils\basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



c:\Users\JSpan\AppData\Local\anaconda3\envs\waterleiden\Lib\site-packages\_plotly_utils\basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [25]:
df_fact_sensor

,gateway_receive_time,device,value
0,2024-10-30 09:48:03,405,20.71
1,2024-10-30 09:13:11,361,7.59
2,2024-10-30 08:55:33,400,23.05
3,2024-10-30 08:41:27,397,11.73
4,2024-10-30 08:27:30,396,11.45
...,...,...,...
61291,2024-10-30 20:08:12,416,10.73
61292,2024-10-30 16:08:48,416,10.76
61293,2024-10-30 12:09:24,416,10.76
61294,2024-11-06 10:19:27,360,39.59


In [26]:
latest_values = df_fact_sensor.merge(df_dim_sensor, how='left', left_on='device', right_on='device_id')
latest_values = latest_values.loc[latest_values.groupby('device')['gateway_receive_time'].idxmax()]
print(len(latest_values))

latest_values[['device_name_org','device','gateway_receive_time']].sort_values(by=['gateway_receive_time'], ascending=False)

26


,device_name_org,device,gateway_receive_time
109693,FF2 0-0144,369,2024-11-06 10:46:20
109691,FF2 0-0135,360,2024-11-06 10:19:27
108981,FF2 0-0131,356,2024-11-06 08:29:19
109172,FF2 0-0137,362,2024-11-06 08:27:48
109090,FF2 0-0136,361,2024-11-06 07:51:57
108809,FF2 0-0102,208,2024-11-06 07:51:10
109458,FF2 0-0173,396,2024-11-06 07:45:40
109611,FF2 0-0191,416,2024-11-06 07:43:56
109376,FF2 0-0171,400,2024-11-06 07:13:24
109526,FF2 0-0179,404,2024-11-06 04:39:56


### 2. Prepareer data
- 2.1 Filter oude locaties er uit
- 2.2 Koppel df_dim_sensor aan df_fact_sensor
- 2.3 Filter per locatie op begin datum plaatsing tot eind datum plaatsing. Locaties zonder einddatum krijgen datum in de toekomst, zodat alle data tot aan vandaag in de dataset zit.
- 2.4 Resample per locatie de tijdreekst op dagniveau. De sensoren geven meerdere sensorwaardes per dag (in principe elke 4 uur), maar dit gaat niet parallel over alle sensoren en voor elke sensor even consequent. Daarom wordt er per dag een gemiddelde, min en max waarde berekend. Daar gaan we de analyse mee doen. De tijdbox functie (add_6H_timebox) gebruiken we momenteel niet, omdat we op dagniveau gaan kijken. Mochten we goede weerdata en sensordata op (4-)uurniveau hebben, dan zouden we ook op een lagere datum_tijd granulariteit analyses kunnen doen.
- 2.5 Prepareer weerdata. De temperatuur en neerslag wordt van weerstation **Zusterhof** in Leiden gebruikt. De overige weerdata vanuit het KNMI van weerstation **Schiphol**. Wanneer de data uit Zusterhof NaN values bevat, wordt deze vervangen door data vanuit het KNMI. **Let op**: we vervangen de negatieve neerslagwaarden (-1) door 0. KNMI maakt onderscheid tussen 'geen regen' (= 0) en 'bijna geen regen' (= -1). In ons geval zien we 'bijna geen' regen als 'geen regen' 
- 2.6 Koppel weerdata aan sensordata
- 2.7 Toevoegen van berekende kolommen

In [27]:
# 2.1 Filter oude locaties er uit
df_dim_sensor = df_dim_sensor[df_dim_sensor['old'] == 'No']
df_dim_sensor.head()

,locatie,device_name,datum_geplaatst,datum_weggehaald,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old,current_location,device_name_org,device_id
0,Arendshorst_30,FF2 0-0193,2023-08-02,2262-04-11,Boom,> 1 jaar,30 cm,groen,Noord,No,Yes,FF2 0-0193,418
2,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372
3,Bachstraat (2)_15,FF2 0-0147,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0147,395
4,Beethovenpark_30,FF1 0-0027,2023-07-04,2023-08-07,Plantenvak,0-1 jaar,30 cm,groen,Zuid,No,No,FF1 0-0027,54
5,Beethovenpark_15,FF1 0-0027,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,groen,Zuid,No,Yes,FF1 0-0027,54


In [28]:
# 2.2 Koppel df_dim_sensor aan df_fact_sensor
df = df_dim_sensor.merge(df_fact_sensor, how='left', left_on=[
                         'device_id'], right_on=['device']).drop(columns='device')

In [29]:
# 2.3 Filter per locatie op begin datum plaatsing tot eind datum plaatsing.

# Filter time series tussen 'datum_geplaatst' en 'datum_opgehaald'
df = df[(df['gateway_receive_time'] > df['datum_geplaatst']) &
        (df['gateway_receive_time'] < df['datum_weggehaald'])]
df.drop(columns=['device_id', 'datum_geplaatst',
        'datum_weggehaald'], inplace=True)

# Rename columns
df.columns = ['locatie', 'device_name', 'soort_plant', 'leeftijd_plant', 'diepte_plaatsing', 'extra_omschrijving', 'locatie_regio',
              'old', 'current_location', 'device_name_org', 'datum_tijd', 'meetwaarde']

# Verzamel locaties
locaties_lst = list(df['locatie'].unique())

# Inspecteer nieuwe dataset
display(df.info(5))
display(df.head(5))
display(df.tail(5))

<class 'pandas.core.frame.DataFrame'>
Index: 42632 entries, 0 to 65762
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   locatie             42632 non-null  object        
 1   device_name         42632 non-null  object        
 2   soort_plant         42632 non-null  object        
 3   leeftijd_plant      42632 non-null  object        
 4   diepte_plaatsing    42632 non-null  object        
 5   extra_omschrijving  42632 non-null  object        
 6   locatie_regio       42632 non-null  object        
 7   old                 42632 non-null  object        
 8   current_location    42632 non-null  object        
 9   device_name_org     42632 non-null  object        
 10  datum_tijd          42632 non-null  datetime64[ns]
 11  meetwaarde          42632 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(10)
memory usage: 4.2+ MB


None

,locatie,device_name,soort_plant,leeftijd_plant,diepte_plaatsing,extra_omschrijving,locatie_regio,old,current_location,device_name_org,datum_tijd,meetwaarde
0,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,Yes,FF2 0-0193,2024-10-02 03:23:52,10.24
1,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,Yes,FF2 0-0193,2024-09-23 05:36:08,7.64
2,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,Yes,FF2 0-0193,2024-09-23 01:38:30,7.69
3,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,Yes,FF2 0-0193,2024-09-22 05:50:16,7.71
4,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,Yes,FF2 0-0193,2024-09-21 21:54:59,7.78


,locatie,device_name,soort_plant,leeftijd_plant,diepte_plaatsing,extra_omschrijving,locatie_regio,old,current_location,device_name_org,datum_tijd,meetwaarde
65758,Vijf meilaan (2)_30,FF1 0-0004,Plantenvak,0-1 jaar,30 cm,bestrating,Zuid,No,Yes,FF1 0-0004,2023-07-04 20:19:33,21.56
65759,Vijf meilaan (2)_30,FF1 0-0004,Plantenvak,0-1 jaar,30 cm,bestrating,Zuid,No,Yes,FF1 0-0004,2023-07-04 12:30:28,25.50
65760,Vijf meilaan (2)_30,FF1 0-0004,Plantenvak,0-1 jaar,30 cm,bestrating,Zuid,No,Yes,FF1 0-0004,2023-07-04 10:16:35,26.22
65761,Vijf meilaan (2)_30,FF1 0-0004,Plantenvak,0-1 jaar,30 cm,bestrating,Zuid,No,Yes,FF1 0-0004,2023-07-04 09:58:44,0.88
65762,Vijf meilaan (2)_30,FF1 0-0004,Plantenvak,0-1 jaar,30 cm,bestrating,Zuid,No,Yes,FF1 0-0004,2023-07-04 07:44:58,1.02


In [30]:
# 2.4 Resample per locatie de tijdreekst op dagniveau.

# Resample time series op dagniveau
df_resample = df.set_index(['locatie', 'datum_tijd'])
resample_freq = 'D'  # '6H'

df = resample(df_resample, resample_freq)
df.columns = ['locatie', 'datum', 'min_meetwaarde',
              'max_meetwaarde', 'gem_meetwaarde']

df = df_dim_sensor[df_dim_sensor['locatie'].isin(locaties_lst)].merge(
    df, how='left', left_on=['locatie'], right_on=['locatie'])
df = df[(df['datum'] > df['datum_geplaatst']) &
        (df['datum'] < df['datum_weggehaald'])]

test = df.copy()

df = df[['locatie', 'diepte_plaatsing', 'leeftijd_plant', 'soort_plant', 'locatie_regio',
       'omgevingsfactoren', 'datum','min_meetwaarde', 'max_meetwaarde', 'gem_meetwaarde']]
       
df.head()

c:\Users\JSpan\OneDrive - ilionx Group BV\03 Projects\17 - Gemeente Leiden - water\waterleiden\custom_functions\data_transformations.py:44: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



,locatie,diepte_plaatsing,leeftijd_plant,soort_plant,locatie_regio,omgevingsfactoren,datum,min_meetwaarde,max_meetwaarde,gem_meetwaarde
1,Arendshorst_30,30 cm,> 1 jaar,Boom,Noord,groen,2023-08-03,9.84,11.10,10.470000
2,Arendshorst_30,30 cm,> 1 jaar,Boom,Noord,groen,2023-08-04,8.87,9.33,9.085000
3,Arendshorst_30,30 cm,> 1 jaar,Boom,Noord,groen,2023-08-05,8.49,8.78,8.647500
4,Arendshorst_30,30 cm,> 1 jaar,Boom,Noord,groen,2023-08-06,9.21,10.13,9.753333
5,Arendshorst_30,30 cm,> 1 jaar,Boom,Noord,groen,2023-08-07,9.31,9.59,9.460000


In [31]:
test[test['locatie'] == 'Bachstraat (1)_15']

,locatie,device_name,datum_geplaatst,datum_weggehaald,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old,current_location,device_name_org,device_id,datum,min_meetwaarde,max_meetwaarde,gem_meetwaarde
429,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2023-08-08,16.77,17.29,17.006667
430,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2023-08-09,16.67,16.91,16.792500
431,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2023-08-10,15.95,16.21,16.080000
432,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2023-08-11,15.60,15.95,15.785000
433,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2023-08-12,15.43,15.95,15.704000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
881,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2024-11-02,8.85,8.94,8.885000
882,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2024-11-03,8.82,8.92,8.872000
883,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2024-11-04,8.82,8.90,8.852000
884,Bachstraat (1)_15,FF2 0-0148,2023-08-07,2262-04-11,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No,Yes,FF2 0-0148,372,2024-11-05,8.78,8.85,8.838333


In [32]:
# # 2.5 Prepareer weerdata
# df_weerdata = df_KNMI.merge(df_weerstation_leiden,
#                             how='left', left_on='datum', right_on='datum')

# # df_weerstation_leiden bevat soms NaN values, deze vervangen we door data uit KNMI
# df_weerdata['temperatuur'] = np.where(df_weerdata['temperatuur_y'].isna(
# ), df_weerdata['temperatuur_x']/10, df_weerdata['temperatuur_y'])
# df_weerdata['neerslag'] = np.where(df_weerdata['neerslag_y'].isna(
# ), df_weerdata['neerslag_x']/10, df_weerdata['neerslag_y'])
# df_weerdata.drop(columns=['temperatuur_x', 'temperatuur_y',
#                  'neerslag_x', 'neerslag_y'], inplace=True)

# # Replace -0.1 values met 0. KNMI maakt onderscheid tussen 'geen regen' (= 0) en 'bijna geen regen' (= -0.1). In ons geval zien we 'bijna geen' regen als 'geen regen' 
# df_weerdata.loc[df_weerdata['neerslag'] < 0, 'neerslag'] = 0

# df_weerdata

In [33]:
# Step 1: Filter df_KNMI to include only data after January 1, 2022
df_weerdata = df_KNMI[df_KNMI['datum'] > '2022-01-01']

# Step 2: Replace NaN values in 'temperatuur' and 'neerslag' columns
df_weerdata['temperatuur'] = np.where(df_weerdata['temperatuur'].isna(), 0, df_weerdata['temperatuur'] / 10)

df_weerdata['neerslag'] = np.where(df_weerdata['neerslag'].isna(), 0, df_weerdata['neerslag'] / 10)


# Step 3: Replace -0.1 values in 'neerslag' column with 0
df_weerdata.loc[df_weerdata['neerslag'] < 0, 'neerslag'] = 0

# Display the modified df_weerdata to user (this would be for testing in an actual use case)
df_weerdata.head()

C:\Users\JSpan\AppData\Local\Temp\ipykernel_16704\2901710040.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\JSpan\AppData\Local\Temp\ipykernel_16704\2901710040.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,datum,windsnelheid,temperatuur,zonneschijn_duur,neerslag_duur,neerslag,luchtdruk,bewolking,vochtigheid,verdamping
25934,2022-01-02,90,11.4,20.0,51.0,6.5,10101,7.0,85.0,3.0
25935,2022-01-03,85,9.4,30.0,0.0,0.0,10066,5.0,84.0,4.0
25936,2022-01-04,45,6.4,5.0,9.0,0.7,9977,7.0,84.0,2.0
25937,2022-01-05,88,5.5,15.0,45.0,3.5,10062,6.0,80.0,2.0
25938,2022-01-06,43,3.3,46.0,10.0,0.2,10165,5.0,84.0,4.0


In [34]:
# # Step 1: Filter df_KNMI to include only data after January 1, 2022
# df_KNMI_filtered = df_KNMI[df_KNMI['datum'] > '2022-01-01']

# # Step 2: Merge df_KNMI_filtered with df_weerstation_leiden on 'datum' column
# df_weerdata = df_KNMI_filtered.merge(df_weerstation_leiden, how='left', on='datum')

# # Step 3: Replace NaN values in 'temperatuur' and 'neerslag' columns
# # 'temperatuur_y' and 'neerslag_y' from 'df_weerstation_leiden', and 'temperatuur_x' and 'neerslag_x' from 'df_KNMI'
# df_weerdata['temperatuur'] = np.where(df_weerdata['temperatuur_y'].isna(),
#                                       df_weerdata['temperatuur_x'] / 10,
#                                       df_weerdata['temperatuur_y'])
# df_weerdata['neerslag'] = np.where(df_weerdata['neerslag_y'].isna(),
#                                    df_weerdata['neerslag_x'] / 10,
#                                    df_weerdata['neerslag_y'])

# # Step 4: Drop the unnecessary original columns
# df_weerdata.drop(columns=['temperatuur_x', 'temperatuur_y', 'neerslag_x', 'neerslag_y'], inplace=True)

# # Step 5: Replace -0.1 values in 'neerslag' column with 0
# df_weerdata.loc[df_weerdata['neerslag'] < 0, 'neerslag'] = 0

# # Display the modified df_weerdata to user (this would be for testing in an actual use case)
# df_weerdata.head()

In [35]:
#2.6 koppel weerdata aan sensordata
df = df.merge(df_weerdata)

# De locaties worden omgezet naar getallen, deze anonieme dataset wordt opgeslagen als 'dataframe.xlsx' 
# en kan worden gebruikt in ChatGPT code interpreter. 
# Daarna worden de locatienamen weer teruggezet naar hun oorspronkelijke namen.
anonymize_location(df)

df.head(5)

,locatie,diepte_plaatsing,leeftijd_plant,soort_plant,locatie_regio,omgevingsfactoren,datum,gem_meetwaarde,windsnelheid,temperatuur,zonneschijn_duur,neerslag_duur,neerslag,luchtdruk,bewolking,vochtigheid,verdamping
0,Arendshorst_30,30 cm,> 1 jaar,Boom,Noord,groen,2023-08-03,10.470000,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0
1,Beethovenpark_30,30 cm,0-1 jaar,Plantenvak,Zuid,groen,2023-08-03,13.341818,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0
2,Beuk vismarkt_binnen_30,30 cm,0-1 jaar,Boom,Binnenstad,bebouwing,2023-08-03,36.860000,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0
3,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-08-03,11.375000,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0
4,Burggravenlaan_15,15 cm,> 1 jaar,Plantenvak,Binnenstad,groen,2023-08-03,16.440000,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0


In [36]:
df.rename(columns={'omgevingsfactoren': 'extra_omschrijving'}, inplace=True)
df.to_excel('./data/prepped_sensor_data.xlsx', index=False)
df.head(5)

,locatie,diepte_plaatsing,leeftijd_plant,soort_plant,locatie_regio,extra_omschrijving,datum,gem_meetwaarde,windsnelheid,temperatuur,zonneschijn_duur,neerslag_duur,neerslag,luchtdruk,bewolking,vochtigheid,verdamping
0,Arendshorst_30,30 cm,> 1 jaar,Boom,Noord,groen,2023-08-03,10.470000,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0
1,Beethovenpark_30,30 cm,0-1 jaar,Plantenvak,Zuid,groen,2023-08-03,13.341818,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0
2,Beuk vismarkt_binnen_30,30 cm,0-1 jaar,Boom,Binnenstad,bebouwing,2023-08-03,36.860000,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0
3,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-08-03,11.375000,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0
4,Burggravenlaan_15,15 cm,> 1 jaar,Plantenvak,Binnenstad,groen,2023-08-03,16.440000,60,17.2,44.0,15.0,1.3,10000,8.0,84.0,21.0


In [37]:
df_KNMI.to_excel('./data/KNMI.xlsx', index=False)
df_KNMI.head(5)

PermissionError: [Errno 13] Permission denied: './data/KNMI.xlsx'